In [0]:
spark.sql("USE lms_analytics")

In [0]:
silver_enrolment_df = spark.table("lms_analytics.bronze_enrolment_activity")

In [0]:
silver_courses_df = spark.table("lms_analytics.bronze_courses")

In [0]:
silver_learners_df = spark.table("lms_analytics.bronze_learners")

In [0]:
print("Enrolment records:", silver_enrolment_df.count())
print("Course records:", silver_courses_df.count())
print("Learner records:", silver_learners_df.count())

In [0]:
silver_enrolment_df = silver_enrolment_df.dropDuplicates(["enrolment_id"])

In [0]:
print("Records after deduplication:", silver_enrolment_df.count())

In [0]:
from pyspark.sql import functions as F

In [0]:
print("Null instructor names:", silver_courses_df.filter(F.col("instructor_name").isNull()).count())

In [0]:
silver_courses_df = silver_courses_df.withColumn(
    "instructor_name",
    F.when(
        F.col("instructor_name").isNull(),
        F.col("instructor_id")
    ).otherwise(F.col("instructor_name"))
)

In [0]:
print("Null instructor names:", silver_courses_df.filter(F.col("instructor_name").isNull()).count())

In [0]:
display(silver_enrolment_df.limit(5))

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "enrol_date",
    F.to_date("enrol_date")
)

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "expected_completion_date",
    F.to_date("expected_completion_date")
)

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "actual_completion_date",
    F.to_date("actual_completion_date")
)

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "last_activity_date",
    F.to_date("last_activity_date")
)

In [0]:
silver_enrolment_df = silver_enrolment_df.withColumn(
    "learning_duration_days",
    F.when(
        F.col("actual_completion_date").isNotNull(),
        F.datediff(
            F.col("actual_completion_date"),
            F.col("enrol_date")
        )
    )
)

In [0]:
silver_enriched_df = silver_enrolment_df.alias("e").join(
    silver_courses_df.alias("c"),
    F.col("e.course_id") == F.col("c.course_id"),
    "left"
)

In [0]:
silver_enriched_df = silver_enriched_df.join(
    silver_learners_df.alias("l"),
    F.col("e.learner_id") == F.col("l.learner_id"),
    "left"
)

In [0]:
print("Silver enriched records:", silver_enriched_df.count())

In [0]:
display(silver_enriched_df.limit(10))

In [0]:
print("Null learner names:", silver_enriched_df.filter(F.col("learner_name").isNull()).count())

In [0]:
print("Null course titles:", silver_enriched_df.filter(F.col("course_title").isNull()).count())

In [0]:
print("Duplicate enrolment IDs:", silver_enriched_df.groupBy("enrolment_id").count().filter(F.col("count") > 1).count())

In [0]:
silver_enriched_df = silver_enrolment_df.join(
    silver_courses_df,
    on="course_id",
    how="left"
)

In [0]:
silver_enriched_df = silver_enriched_df.join(
    silver_learners_df,
    on="learner_id",
    how="left"
)

In [0]:
silver_enriched_df = silver_enriched_df.select(
    "enrolment_id",
    "learner_id",
    "learner_name",
    "email",
    "city",
    "subscription_type",
    "course_id",
    "course_title",
    "category",
    "instructor_id",
    "instructor_name",
    "duration_hours",
    "difficulty_level",
    "price_inr",
    "enrol_date",
    "expected_completion_date",
    "actual_completion_date",
    "status",
    "progress_pct",
    "last_activity_date",
    "assessment_score",
    "attempts",
    "feedback_rating",
    "certificate_issued",
    "learning_duration_days"
)

In [0]:
print("Silver enriched records:", silver_enriched_df.count())

In [0]:
silver_enriched_df.write.format("delta").mode("overwrite").saveAsTable(
    "lms_analytics.silver_enriched_enrolments"
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS silver_records
FROM lms_analytics.silver_enriched_enrolments
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS duplicate_enrolment_ids
FROM (
    SELECT enrolment_id
    FROM lms_analytics.silver_enriched_enrolments
    GROUP BY enrolment_id
    HAVING COUNT(*) > 1
)
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS null_learner_names
FROM lms_analytics.silver_enriched_enrolments
WHERE learner_name IS NULL
""").show()

In [0]:
display(spark.sql("""
SELECT *
FROM lms_analytics.silver_enriched_enrolments
LIMIT 10
"""))

In [0]:
spark.sql("""
SELECT status, COUNT(*) AS total_records
FROM lms_analytics.silver_enriched_enrolments
GROUP BY status
ORDER BY total_records DESC
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS invalid_progress
FROM lms_analytics.silver_enriched_enrolments
WHERE progress_pct < 0 OR progress_pct > 100
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS invalid_assessment_scores
FROM lms_analytics.silver_enriched_enrolments
WHERE assessment_score < 0 OR assessment_score > 100
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS invalid_feedback_ratings
FROM lms_analytics.silver_enriched_enrolments
WHERE feedback_rating < 1 OR feedback_rating > 5
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS invalid_dates
FROM lms_analytics.silver_enriched_enrolments
WHERE actual_completion_date IS NOT NULL
AND actual_completion_date < enrol_date
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS null_last_activity_dates
FROM lms_analytics.silver_enriched_enrolments
WHERE last_activity_date IS NULL
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS null_assessment_scores
FROM lms_analytics.silver_enriched_enrolments
WHERE assessment_score IS NULL
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS null_learning_duration
FROM lms_analytics.silver_enriched_enrolments
WHERE learning_duration_days IS NULL
""").show()

In [0]:
spark.sql("""
SELECT COUNT(*) AS completed_records_with_null_completion_date
FROM lms_analytics.silver_enriched_enrolments
WHERE status = 'Completed'
AND actual_completion_date IS NULL
""").show()

In [0]:
spark.sql("SHOW TABLES IN lms_analytics").show()